In [1]:
import pandas as pd

import matplotlib.pyplot as plt
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import precision_score, recall_score, accuracy_score

# --- 1. Load the Dataset ---
# Change the file path to point to a training dataset that contains the 'Loan_Status' column.
# Example: `train_data = pd.read_csv("path/to/your/train_dataset.csv")`
try:
    train_data = pd.read_csv("train_u6lujuX_CVtuZ9i.csv")
    print("Dataset loaded successfully.")
except FileNotFoundError:
    print("Error: The file was not found. Please check your file path.")
    # Exit the script if the file is not found
    exit()

# --- 2. Data Cleaning ---
# Drop the "Loan_ID" and "Dependents" columns as per your original code.
# The `inplace=True` parameter modifies the DataFrame directly.
try:
    train_data.drop(["Loan_ID", "Dependents"], axis=1, inplace=True)
except KeyError as e:
    print(f"\nWarning: Could not drop columns. The following columns were not found in the DataFrame: {e}")

# Fill missing values for categorical columns using the mode.
cols_to_fill_mode = ["Gender", "Married", "Self_Employed"]
for col in cols_to_fill_mode:
    train_data.loc[:, col] = train_data[col].fillna(train_data[col].mode()[0])

# Fill missing values for numerical columns using the mean.
cols_to_fill_mean = ["LoanAmount", "Loan_Amount_Term", "Credit_History"]
for col in cols_to_fill_mean:
    train_data.loc[:, col] = train_data[col].fillna(train_data[col].mean())

# Verify that all missing values have been filled.
print("\nMissing values after cleaning:")
print(train_data.isnull().sum())

# --- 3. Encoding Categorical Data ---
# Initialize the OrdinalEncoder.
ord_enc = OrdinalEncoder()

# List the columns you want to encode.
# We will check if 'Loan_Status' exists before adding it to the list.
categorical_cols = ["Gender", "Married", "Education", "Self_Employed", "Property_Area"]
if "Loan_Status" in train_data.columns:
    categorical_cols.append("Loan_Status")

# Use .loc to ensure you are modifying the original DataFrame correctly.
train_data.loc[:, categorical_cols] = ord_enc.fit_transform(train_data[categorical_cols])

# Display the first few rows of the updated DataFrame.
print("\nDataFrame after ordinal encoding:")
print(train_data.head())

# --- 4. Train-Test Split ---
# Separate the features (X) and the target variable (y).
# The 'Loan_Status' column is used as the target variable.
if "Loan_Status" in train_data.columns:
    X = train_data.drop("Loan_Status", axis=1)
    # This line is the fix: get values and explicitly cast to int
    y = train_data["Loan_Status"].values.astype(int)
    
    # Split the data into training and testing sets.
    # test_size=0.2 means 20% of the data will be used for testing.
    # random_state=2 ensures the split is consistent.
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

    # Print the shapes of the new datasets to verify the split.
    print("\nTrain-Test Split Shapes:")
    print(f"X_train shape: {X_train.shape}")
    print(f"X_test shape: {X_test.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"y_test shape: {y_test.shape}")

    # --- 5. Model Training and Prediction ---
    # Initialize the Gaussian Naive Bayes classifier
    gfc = GaussianNB()

    # Fit the model to the training data
    gfc.fit(X_train, y_train)

    # Make predictions on the test set
    pred1 = gfc.predict(X_test)
    
    print("\nPredictions on the test set:")
    print(pred1)

    # --- 6. Model Evaluation ---
    # Define a function to evaluate the model's performance
    def loss(y_true, y_pred):
        pre = precision_score(y_true, y_pred, average='weighted', zero_division=0)
        rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
        acc = accuracy_score(y_true, y_pred)
        
        print("\nModel Evaluation:")
        print(f"Precision: {pre}")
        print(f"Recall: {rec}")
        print(f"Accuracy: {acc}")

    # Call the evaluation function with the true labels and predictions
    loss(y_test, pred1)

else:
    print("\nError: The 'Loan_Status' column is not in the DataFrame. "
          "The train-test split and model training cannot be performed without a target variable.")


Dataset loaded successfully.

Missing values after cleaning:
Gender               0
Married              0
Education            0
Self_Employed        0
ApplicantIncome      0
CoapplicantIncome    0
LoanAmount           0
Loan_Amount_Term     0
Credit_History       0
Property_Area        0
Loan_Status          0
dtype: int64

DataFrame after ordinal encoding:
  Gender Married Education Self_Employed  ApplicantIncome  CoapplicantIncome  \
0    1.0     0.0       0.0           0.0             5849                0.0   
1    1.0     1.0       0.0           0.0             4583             1508.0   
2    1.0     1.0       0.0           1.0             3000                0.0   
3    1.0     1.0       1.0           0.0             2583             2358.0   
4    1.0     0.0       0.0           0.0             6000                0.0   

   LoanAmount  Loan_Amount_Term  Credit_History Property_Area Loan_Status  
0  146.412162             360.0             1.0           2.0         1.0  
1  12

In [2]:
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import precision_score, recall_score, accuracy_score
print("\n\n--- Training Support Vector Classifier (SVC) with GridSearchCV ---")
    
    # Initialize the SVC classifier
svc = SVC()
    
    # Define the parameter range for GridSearchCV
param_grid = {'C': [0.1, 1, 10, 100, 1000],
                  'gamma': [1, 0.1, 0.01, 0.001, 0.0001],
                  'kernel': ['rbf']}
    
    # Set up GridSearchCV
grid = GridSearchCV(SVC(), param_grid, refit=True, verbose=3)
    
    # Fit the grid search to the data
grid.fit(X_train, y_train)
    
    # Print the best parameters and the best score
print("\nBest Parameters found by GridSearchCV:")
print(grid.best_params_)
print("\nBest Score found by GridSearchCV:")
print(grid.best_score_)
    
    # Make predictions using the best model from GridSearchCV
grid_predictions = grid.predict(X_test)
    
print("\nSVC Predictions (from best model):")
print(grid_predictions)
    
    # --- 8. Model Evaluation (SVC) ---
print("\n--- SVC Performance (from best model) ---")
loss(y_test, grid_predictions)





--- Training Support Vector Classifier (SVC) with GridSearchCV ---
Fitting 5 folds for each of 25 candidates, totalling 125 fits
[CV 1/5] END ........C=0.1, gamma=1, kernel=rbf;, score=0.687 total time=   0.0s
[CV 2/5] END ........C=0.1, gamma=1, kernel=rbf;, score=0.694 total time=   0.0s
[CV 3/5] END ........C=0.1, gamma=1, kernel=rbf;, score=0.694 total time=   0.0s
[CV 4/5] END ........C=0.1, gamma=1, kernel=rbf;, score=0.684 total time=   0.0s
[CV 5/5] END ........C=0.1, gamma=1, kernel=rbf;, score=0.684 total time=   0.0s
[CV 1/5] END ......C=0.1, gamma=0.1, kernel=rbf;, score=0.687 total time=   0.0s
[CV 2/5] END ......C=0.1, gamma=0.1, kernel=rbf;, score=0.694 total time=   0.0s
[CV 3/5] END ......C=0.1, gamma=0.1, kernel=rbf;, score=0.694 total time=   0.0s
[CV 4/5] END ......C=0.1, gamma=0.1, kernel=rbf;, score=0.684 total time=   0.0s
[CV 5/5] END ......C=0.1, gamma=0.1, kernel=rbf;, score=0.684 total time=   0.0s
[CV 1/5] END .....C=0.1, gamma=0.01, kernel=rbf;, score=0.6

In [3]:
grid.best_params_

{'C': 0.1, 'gamma': 1, 'kernel': 'rbf'}

In [4]:
svc = SVC(C=0.1, gamma=1, kernel='rbf')

# Fit the model to the training data
svc.fit(X_train, y_train)

# Make predictions on the test set
pred2 = svc.predict(X_test)

print("\nSVC Predictions:")
print(pred2)

# --- 2. Model Evaluation (SVC) ---
# Define a function to evaluate the model's performance
def loss(y_true, y_pred):
    pre = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    
    print("\nModel Evaluation:")
    print(f"Precision: {pre}")
    print(f"Recall: {rec}")
    print(f"Accuracy: {acc}")

print("\n--- SVC Performance ---")
loss(y_test, pred2)



SVC Predictions:
[1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1
 1 1 1 1 1 1 1 1 1 1 1 1]

--- SVC Performance ---

Model Evaluation:
Precision: 0.4663890541344438
Recall: 0.6829268292682927
Accuracy: 0.6829268292682927


In [5]:
!pip install xgboost



In [8]:
import xgboost as xgb
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
import joblib
import numpy as np
import pandas as pd

# ---------------------------------------------------------
# 1. Feature Data Conversion (Fixes XGBoost KeyError: 'object')
# ---------------------------------------------------------
# XGBoost requires columns to be 'category' dtype instead of 'object'
X_train_xgb = X_train.copy()
X_test_xgb = X_test.copy()

cat_cols = X_train_xgb.select_dtypes(include=['object']).columns

for col in cat_cols:
    X_train_xgb[col] = X_train_xgb[col].astype('category')
    X_test_xgb[col] = X_test_xgb[col].astype('category')

# Encode target variable (Y/N -> 1/0) if it's text
if hasattr(y_train, 'map') and y_train.dtype == 'object':
    y_train_xgb = y_train.map({'N': 0, 'Y': 1})
    y_test_xgb = y_test.map({'N': 0, 'Y': 1})
else:
    y_train_xgb = y_train
    y_test_xgb = y_test

# ---------------------------------------------------------
# 2. XGBoost Classifier Execution
# ---------------------------------------------------------
xgb_model = xgb.XGBClassifier(enable_categorical=True, random_state=2)
xgb_model.fit(X_train_xgb, y_train_xgb)

pred3 = xgb_model.predict(X_test_xgb)

# Convert predictions back to 'N'/'Y' if target was object
if hasattr(y_train, 'map') and y_train.dtype == 'object':
    pred3 = pd.Series(pred3).map({0: 'N', 1: 'Y'})

print("--- XGBoost Performance ---")
loss(y_test, pred3)

# ---------------------------------------------------------
# 3. Decision Tree Classifier (RandomizedSearchCV)
# ---------------------------------------------------------
# Convert object types to category or factorize for Sklearn Decision Tree
X_train_dt = X_train_xgb.copy()
X_test_dt = X_test_xgb.copy()

for col in cat_cols:
    X_train_dt[col] = X_train_dt[col].cat.codes
    X_test_dt[col] = X_test_dt[col].cat.codes

params = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [None] + list(np.arange(2, 20)),
    'min_samples_split': np.arange(2, 11),
    'min_samples_leaf': np.arange(1, 10)
}

rand_clf = RandomizedSearchCV(
    estimator=DecisionTreeClassifier(random_state=2),
    param_distributions=params,
    n_iter=20,
    cv=5,
    n_jobs=-1,
    random_state=2
)

rand_clf.fit(X_train_dt, y_train)

best_dt = rand_clf.best_estimator_
pred4 = best_dt.predict(X_test_dt)

print("\n--- Decision Tree Performance ---")
loss(y_test, pred4)

# ---------------------------------------------------------
# 4. Save Models
# ---------------------------------------------------------
joblib.dump(best_dt, 'best_model.pkl')
print("\n✓ Best Decision Tree Model saved successfully as best_model.pkl!")

--- XGBoost Performance ---

Model Evaluation:
Precision: 0.7547862575399947
Recall: 0.7642276422764228
Accuracy: 0.7642276422764228

--- Decision Tree Performance ---

Model Evaluation:
Precision: 0.7291399229781772
Recall: 0.7398373983739838
Accuracy: 0.7398373983739838

✓ Best Decision Tree Model saved successfully as best_model.pkl!


In [6]:
# This code assumes you have already run the previous data cleaning and splitting steps
# It uses the X_train, y_train, X_test, and y_test variables from the previous notebook cell.

# To fix the 'ModuleNotFoundError', you need to install the xgboost library.
# Run one of the following commands in your terminal or a new notebook cell:
# For pip: !pip install xgboost
# For conda: !conda install -c anaconda py-xgboost
# You may need to restart your Jupyter kernel after the installation.

# Import the necessary libraries
import xgboost as xgb
from sklearn.metrics import precision_score, recall_score, accuracy_score

# --- 1. Model Training and Prediction (XGBoost) ---
print("\n--- Training XGBoost Classifier ---")

# Separate the categorical and numerical columns
categorical_cols = ["Gender", "Married", "Education", "Self_Employed", "Property_Area"]
numerical_cols = ["ApplicantIncome", "CoapplicantIncome", "LoanAmount", "Loan_Amount_Term", "Credit_History"]

# Convert only the categorical features to 'category' data type for XGBoost
for col in categorical_cols:
    X_train.loc[:, col] = X_train[col].astype('category')
    X_test.loc[:, col] = X_test[col].astype('category')

# Convert numerical features to a numeric type to prevent errors
for col in numerical_cols:
    X_train.loc[:, col] = pd.to_numeric(X_train[col], errors='coerce').astype('float64')
    X_test.loc[:, col] = pd.to_numeric(X_test[col], errors='coerce').astype('float64')

# Initialize the XGBClassifier with the specified parameters and enable_categorical=True
xgb_model = xgb.XGBClassifier(learning_rate=0.1, n_estimators=1000, max_depth=3,
                              min_child_weight=1, gamma=0, subsample=0.8,
                              colsample_bytree=0.8, objective='binary:logistic',
                              nthread=4, scale_pos_weight=1, seed=27,
                              enable_categorical=True)

# Fit the model to the training data
xgb_model.fit(X_train, y_train)

# Make predictions on the test set
pred3 = xgb_model.predict(X_test)

print("\nXGBoost Predictions:")
print(pred3)

# --- 2. Model Evaluation (XGBoost) ---
# Define a function to evaluate the model's performance
def loss(y_true, y_pred):
    pre = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    
    print("\nModel Evaluation:")
    print(f"Precision: {pre}")
    print(f"Recall: {rec}")
    print(f"Accuracy: {acc}")

print("\n--- XGBoost Performance ---")
loss(y_test, pred3)



--- Training XGBoost Classifier ---


ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:Gender: object, Married: object, Education: object, Self_Employed: object, Property_Area: object

In [ ]:
# This code assumes you have already run the previous data cleaning and splitting steps
# It uses the X_train, y_train, X_test, and y_test variables from the previous notebook cell.

# Import the necessary libraries
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score
import numpy as np

# Define a function to perform the randomized search
def randomized_search(params, runs=20, clf=DecisionTreeClassifier(random_state=2)):
    # Initialize the RandomizedSearchCV object
    # n_iter=runs sets the number of parameter settings that are sampled.
    # cv=5 sets the number of folds for cross-validation.
    # n_jobs=-1 uses all available CPU cores.
    # random_state=2 ensures reproducibility.
    rand_clf = RandomizedSearchCV(clf, params, n_iter=runs, cv=5, n_jobs=-1, random_state=2)
    
    # Fit the model to the training data to find the best parameters
    rand_clf.fit(X_train, y_train)
    
    # Get the best estimator (the best model found during the search)
    best_model = rand_clf.best_estimator_
    
    # Extract the best score from the search results
    best_score = rand_clf.best_score_
    
    # Print the training score of the best model
    print("Training score: {:.3f}".format(best_score))
    
    # Use the best model to predict labels on the test set
    y_pred = best_model.predict(X_test)
    
    # Compute and print the accuracy of the predictions on the test set
    accuracy = accuracy_score(y_test, y_pred)
    print('Test score: {:.3f}'.format(accuracy))
    
    # Return the best model found
    return best_model

# Define the parameters to search over for the Decision Tree
dt_params = {
    'max_depth': np.arange(1, 10, 1),
    'max_features': [0.5, 0.7, 1.0],
    'criterion': ['gini', 'entropy'],
    'min_samples_split': [2, 3, 4],
    'min_samples_leaf': [1, 2, 3]
}

# Call the randomized search function with the defined parameters
best_dt_model = randomized_search(dt_params)



In [ ]:
# This code assumes you have already run the previous data cleaning and splitting steps
# It uses the X_train, y_train, X_test, and y_test variables from the previous notebook cell.

# Import the necessary libraries
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import accuracy_score

# --- 1. Model Training and Prediction (Decision Tree with RandomizedSearchCV) ---
print("\n--- Training Decision Tree with RandomizedSearchCV ---")

# Initialize the Decision Tree classifier
clf = DecisionTreeClassifier(random_state=2)

# Define the new, more extensive parameter grid for RandomizedSearchCV
# The value 'auto' has been removed from 'max_features' to prevent a FitFailedWarning.
dt_params = {
    'criterion': ['entropy', 'gini'],
    'splitter': ['random', 'best'],
    'min_weight_fraction_leaf': [0.0, 0.0025, 0.005, 0.0075, 0.01],
    'min_samples_split': [2, 3, 4, 5, 6, 8, 10],
    'min_samples_leaf': [1, 0.01, 0.02, 0.03, 0.04],
    'min_impurity_decrease': [0.0, 0.0005, 0.005, 0.05, 0.10, 0.15, 0.2],
    'max_leaf_nodes': [10, 15, 20, 25, 30, 35, 40, 45, 50, None],
    'max_features': ['sqrt', 0.95, 0.90, 0.85, 0.80, 0.75, 0.70],
    'max_depth': [None, 2, 4, 6, 8]
}

# Initialize the RandomizedSearchCV object
# n_iter=20 sets the number of parameter settings that are sampled.
# cv=5 sets the number of folds for cross-validation.
# n_jobs=-1 uses all available CPU cores.
# random_state=2 ensures reproducibility.
rand_clf = RandomizedSearchCV(clf, dt_params, n_iter=20, cv=5, n_jobs=-1, random_state=2)

# Fit the grid search to the data
rand_clf.fit(X_train, y_train)

# Get the best estimator (the best model found during the search)
best_model = rand_clf.best_estimator_

# Extract the best score from the search results
best_score = rand_clf.best_score_

# Print the training score of the best model
print("Training score: {:.3f}".format(best_score))

# Use the best model to predict labels on the test set
y_pred = best_model.predict(X_test)

# Compute and print the accuracy of the predictions on the test set
accuracy = accuracy_score(y_test, y_pred)
print('Test score: {:.3f}'.format(accuracy))

# --- 2. Print Best Parameters ---
print("\nBest Parameters found by RandomizedSearchCV:")
print(rand_clf.best_params_)


In [ ]:
# --- 2. Use a Pre-tuned Decision Tree Classifier ---
print("\n--- Using a Pre-tuned Decision Tree Classifier ---")

# Initialize a new DecisionTreeClassifier with the specific parameters from the image
ds = DecisionTreeClassifier(
    max_depth=8,
    max_features=0.9,
    max_leaf_nodes=30,
    min_impurity_decrease=0.05,
    min_samples_leaf=0.02,
    min_samples_split=10,
    min_weight_fraction_leaf=0.005,
    random_state=2,
    splitter='random'
)

# Fit the new model to the training data
ds.fit(X_train, y_train)

# Make predictions on the test set
pred4 = ds.predict(X_test)

# Evaluate the model using the previously defined 'loss' function
print("\nModel Evaluation (Pre-tuned Decision Tree):")
loss(y_test, pred4)


In [ ]:
# --- 3. Save and Load the Model ---
import joblib
print("\n--- Saving and Loading the Model ---")

# Save the trained model to a file named 'model.pkl'
joblib.dump(ds, "model.pkl")
print("Model saved as 'model.pkl'")

# Load the saved model from the file
loaded_model = joblib.load('model.pkl')
print("Model loaded from 'model.pkl'")

# You can now use the loaded model to make predictions
loaded_predictions = loaded_model.predict(X_test)
print("\nPredictions from the loaded model:")
print(loaded_predictions)
